In [1]:
!pip install --upgrade pip
!pip install torch-summary
!pip install arabert peft transformers
!pip install git+https://github.com/ARBML/rouge_score_ar
!pip install accelerate -U
!pip install sentencepiece 
!pip install lxml 
!pip install keras-nlp
!pip install gdown datasets
!pip install bitsandbytes peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.7 MB/s eta 0:00:0000:010:01
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Uninstalling pip-23.0.1:
      Successfully uninstalled pip-23.0.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 4.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 56.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 16.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 16.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 22.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.3/772.3 kB 36.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 72.6 MB

In [2]:
5

5

In [3]:
import tensorflow as tf

D0709 00:42:48.072709641      15 config.cc:119]                        gRPC EXPERIMENT tcp_frame_size_tuning               OFF (default:OFF)
D0709 00:42:48.072739478      15 config.cc:119]                        gRPC EXPERIMENT tcp_rcv_lowat                       OFF (default:OFF)
D0709 00:42:48.072742889      15 config.cc:119]                        gRPC EXPERIMENT peer_state_based_framing            OFF (default:OFF)
D0709 00:42:48.072745470      15 config.cc:119]                        gRPC EXPERIMENT flow_control_fixes                  ON  (default:ON)
D0709 00:42:48.072747741      15 config.cc:119]                        gRPC EXPERIMENT memory_pressure_controller          OFF (default:OFF)
D0709 00:42:48.072750060      15 config.cc:119]                        gRPC EXPERIMENT unconstrained_max_quota_buffer_size OFF (default:OFF)
D0709 00:42:48.072753446      15 config.cc:119]                        gRPC EXPERIMENT new_hpack_huffman_decoder           ON  (default:ON)
D0709 00:42:48.

In [4]:

tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect(tpu="local") # "local" for 1VM TPU

strategy = tf.distribute.TPUStrategy(tpu)
print("RUNNING TPU")

    
print("REPLICAS: ", strategy.num_replicas_in_sync)

INFO:tensorflow:Deallocate tpu buffers before initializing tpu system.
INFO:tensorflow:Initializing the TPU system: local
INFO:tensorflow:Finished initializing TPU system.
INFO:tensorflow:Found TPU system:
INFO:tensorflow:*** Num TPU Cores: 8
INFO:tensorflow:*** Num TPU Workers: 1
INFO:tensorflow:*** Num TPU Cores Per Worker: 8
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:CPU:0, CPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:0, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:1, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:2, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:3, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:4, TPU

In [5]:
def map_to(batch):

    model_inputs = tokenizer(batch['article'],
                             padding="max_length",
                             truncation=True,
                              max_length=1000, return_tensors='np')

    labels = tokenizer(batch['summary'], text_target=batch['summary'],
                       padding="max_length",
                       truncation=True,
                       max_length=400, return_tensors='np')
    
    model_inputs['decoder_input_ids'] = labels['input_ids']
    model_inputs['decoder_attention_mask'] = labels['attention_mask']
    model_inputs["labels"] = labels["labels"]
    
    return model_inputs

def map_to_val(batch):

    model_inputs = tokenizer(batch['document'],
                             padding="max_length",
                             truncation=True,
                              max_length=1000, return_tensors='np')

    labels = tokenizer(batch['summary'], text_target=batch['summary'],
                       padding="max_length",
                       truncation=True,
                       max_length=400, return_tensors='np') 
    
    model_inputs['decoder_input_ids'] = labels['input_ids']
    model_inputs['decoder_attention_mask'] = labels['attention_mask']
    model_inputs["labels"] = labels["labels"]
    
    return model_inputs


In [6]:
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM, AutoModelForSeq2SeqLM, MBart50TokenizerFast
# from peft import LoraConfig, get_peft_model, TaskType, PrefixTuningConfig, prepare_model_for_int8_training
import torch
from datasets import load_dataset, DatasetDict
from transformers import DataCollatorForSeq2Seq
from keras_nlp.metrics import Perplexity


/usr/local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
model_id = "facebook/mbart-large-50"


tokenizer = MBart50TokenizerFast.from_pretrained(model_id, src_lang="ar_AR", tgt_lang="ar_AR")
with strategy.scope():
    model = TFAutoModelForSeq2SeqLM.from_pretrained(model_id,)
    opt = tf.keras.optimizers.AdamW(1e-6, 0.01, global_clipnorm=0.5)
    per = Perplexity(from_logits=True, name="perplexity", mask_token_id=tokenizer.pad_token_id,)
    for i, l in enumerate(model.layers[0].encoder.layers):
        if i < 8:
            l.trainable = False
        
    for i, l in enumerate(model.layers[0].decoder.layers):
        if i < 2:
            l.trainable = False
            
    model.compile(optimizer=opt, metrics=[per])

All model checkpoint layers were used when initializing TFMBartForConditionalGeneration.

All the layers of TFMBartForConditionalGeneration were initialized from the model checkpoint at facebook/mbart-large-50.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMBartForConditionalGeneration for predictions without further training.


In [8]:
model.summary()

Model: "tfm_bart_for_conditional_generation"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 model (TFMBartMainLayer)    multiple                  610879488 
                                                                 
 final_logits_bias (BiasLaye  multiple                 250054    
 r)                                                              
                                                                 
Total params: 611,129,542
Trainable params: 476,516,352
Non-trainable params: 134,613,190
_________________________________________________________________


In [9]:
import os
path = '/kaggle/input/arabictextsummarization'
train_files = [
                 '/kaggle/input/arabictextsummarization/EGY_alyoomSabi_news_1.csv',
                 '/kaggle/input/arabictextsummarization/PAL_alquds_news.csv',
                 '/kaggle/input/arabictextsummarization/نسخة من UAE_bayan_news_1.csv'
                ]
val_files = ['/kaggle/input/arabic-text-summarization/data/labeld_valid.csv',
#                               '/kaggle/input/arabictextsummarization/BAH_waseet_news_1.csv',
                             ]

In [10]:
data = load_dataset('csv', data_files=train_files,)
val = load_dataset('csv', data_files=val_files,)

Extracting data files: 100%|██████████| 1/1 [00:00<00:00, 12.45it/s]


Dataset csv downloaded and prepared to /root/.cache/huggingface/datasets/csv/default-ad8e49c4c6484116/0.0.0/eea64c71ca8b46dd3f537ed218fc9bf495d5707789152eb2764f5c78fa66d59d. Subsequent calls will reuse this data.


100%|██████████| 1/1 [00:00<00:00, 31.87it/s]


Extracting data files: 100%|██████████| 1/1 [00:00<00:00, 54.72it/s]


Dataset csv downloaded and prepared to /root/.cache/huggingface/datasets/csv/default-737d32ff6d585257/0.0.0/eea64c71ca8b46dd3f537ed218fc9bf495d5707789152eb2764f5c78fa66d59d. Subsequent calls will reuse this data.


100%|██████████| 1/1 [00:00<00:00, 470.00it/s]


In [11]:
data

DatasetDict({
    train: Dataset({
        features: ['item', 'address', 'title', 'article', 'summary'],
        num_rows: 1161417
    })
})

In [12]:
val

DatasetDict({
    train: Dataset({
        features: ['example_id', 'document', 'summary'],
        num_rows: 154
    })
})

In [13]:
import gc
gc.collect()

41

In [14]:
train_dataset = data.shuffle(seed=36).map(map_to, batched=True, batch_size=20, drop_last_batch=True,
                                                   num_proc=25, remove_columns=['item', 'address', 'title', 'article', 'summary'])

valid_dataset = val.shuffle(seed=36).map(map_to_val, batched=True, batch_size=22, drop_last_batch=True, num_proc=1,
                                                        remove_columns=['example_id', 'document', 'summary'])

In [15]:
train_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'decoder_input_ids', 'decoder_attention_mask', 'labels'],
        num_rows: 1161000
    })
})

In [16]:
valid_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'decoder_input_ids', 'decoder_attention_mask', 'labels'],
        num_rows: 154
    })
})

In [17]:
import torch
import gc
torch.cuda.empty_cache()
gc.collect(0)
gc.collect(1)
gc.collect()

0

In [19]:
tf_train_dataset = model.prepare_tf_dataset(train_dataset['train'],
                                            batch_size=32, drop_remainder=True,
                                            shuffle=True)

tf_dev_dataset = model.prepare_tf_dataset(valid_dataset['train'],
                                        batch_size=22, drop_remainder=True,
                                        shuffle=False)

In [20]:
model.fit(tf_train_dataset,validation_data=tf_dev_dataset, epochs=4, steps_per_epoch=1000)

Epoch 1/4


/usr/local/lib/python3.8/site-packages/tensorflow/python/framework/indexed_slices.py:459: UserWarning: Converting sparse IndexedSlices to a dense Tensor with 256055296 elements. This may consume a large amount of memory.
  warnings.warn(
2023-07-09 00:51:46.136547: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.
2023-07-09 00:51:48.278813: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.


1000/1000 [==============================] - ETA: 0s - loss: 1.3346 - perplexity: 3.7983

2023-07-09 01:01:54.613167: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.
2023-07-09 01:01:55.544849: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node AssignAddVariableOp.


1000/1000 [==============================] - 714s 538ms/step - loss: 1.3346 - perplexity: 3.7983 - val_loss: 0.1674 - val_perplexity: 1.1816
Epoch 2/4
1000/1000 [==============================] - 500s 500ms/step - loss: 0.0622 - perplexity: 1.0642 - val_loss: 0.0058 - val_perplexity: 1.0055
Epoch 3/4
1000/1000 [==============================] - 501s 501ms/step - loss: 0.0016 - perplexity: 1.0016 - val_loss: 0.0032 - val_perplexity: 1.0031
Epoch 4/4
1000/1000 [==============================] - 500s 500ms/step - loss: 9.0635e-04 - perplexity: 1.0009 - val_loss: 0.0034 - val_perplexity: 1.0034


In [22]:
version_name = 'version-1000'
model.save_pretrained(version_name)
pt_model = AutoModelForSeq2SeqLM.from_pretrained(version_name, from_tf=True)
pt_model.save_pretrained(version_name)
tokenizer.save_pretrained(version_name)

All TF 2.0 model weights were used when initializing MBartForConditionalGeneration.

All the weights of MBartForConditionalGeneration were initialized from the TF 2.0 model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use MBartForConditionalGeneration for predictions without further training.


('version-1000/tokenizer_config.json',
 'version-1000/special_tokens_map.json',
 'version-1000/sentencepiece.bpe.model',
 'version-1000/added_tokens.json',
 'version-1000/tokenizer.json')

In [24]:
!ls version-1000

config.json		sentencepiece.bpe.model  tokenizer.json
generation_config.json	special_tokens_map.json  tokenizer_config.json
pytorch_model.bin	tf_model.h5


In [26]:
for i, l in enumerate(model.layers[0].encoder.layers):
    if i < 10:
        l.trainable = False
        
for i, l in enumerate(model.layers[0].decoder.layers):
    if i < 9:
        l.trainable = False
    if i == 0:
        l.trainable = True
model.summary()
            

Model: "tfm_bart_for_conditional_generation"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 model (TFMBartMainLayer)    multiple                  610879488 
                                                                 
 final_logits_bias (BiasLaye  multiple                 250054    
 r)                                                              
                                                                 
Total params: 611,129,542
Trainable params: 350,543,872
Non-trainable params: 260,585,670
_________________________________________________________________


In [27]:
model.fit(tf_train_dataset,validation_data=tf_dev_dataset, epochs=4, steps_per_epoch=100)

Epoch 1/4
20/20 [==============================] - 15s 733ms/step - loss: 0.6319 - perplexity: 1.8812 - val_loss: 0.2415 - val_perplexity: 1.2727
Epoch 2/4
20/20 [==============================] - 15s 732ms/step - loss: 0.6094 - perplexity: 1.8393 - val_loss: 0.2404 - val_perplexity: 1.2712
Epoch 3/4
20/20 [==============================] - 15s 732ms/step - loss: 0.5783 - perplexity: 1.7830 - val_loss: 0.2392 - val_perplexity: 1.2697
Epoch 4/4
20/20 [==============================] - 15s 744ms/step - loss: 0.6096 - perplexity: 1.8396 - val_loss: 0.2380 - val_perplexity: 1.2682


In [28]:
model.save_pretrained('version-500')
pt_model = AutoModelForSeq2SeqLM.from_pretrained('version-500', from_tf=True)
pt_model.save_pretrained('version-500')
tokenizer.save_pretrained('version-500')

All TF 2.0 model weights were used when initializing MBartForConditionalGeneration.

All the weights of MBartForConditionalGeneration were initialized from the TF 2.0 model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use MBartForConditionalGeneration for predictions without further training.


('version-500/tokenizer_config.json',
 'version-500/special_tokens_map.json',
 'version-500/sentencepiece.bpe.model',
 'version-500/added_tokens.json',
 'version-500/tokenizer.json')

# For Inference

In [1]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

peft_model_id = "version-1000"

# load base LLM model and tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(peft_model_id)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)

model.eval()

print("model loaded")


/usr/local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model loaded


In [14]:
pd

<module 'pandas' from '/usr/local/lib/python3.8/site-packages/pandas/__init__.py'>

In [2]:
import pandas as pd
from tqdm import tqdm
from rouge_score import rouge_scorer
5

5

In [3]:
validation = pd.read_csv('/kaggle/input/arabic-text-summarization/data/labeld_valid.csv')

In [4]:
model.device

device(type='cpu')

In [5]:
# @tf.function
def generate_summary_pt(text):
    input_ids = tokenizer(
                        [text],
                        return_tensors="pt",
                        )["input_ids"].to(model.device)
    max_output_length = 250
    output_ids = model.generate(
                                input_ids=input_ids,
                                num_beams=8,
                                length_penalty=2.0,
                                max_length=max_output_length,
                                min_length=max_output_length//2 ,
                                early_stopping=True,
                                num_return_sequences=1,
                                top_p=0.7,
                                top_k=100,
                                temperature=0.5,
                                no_repeat_ngram_size=4,
                                remove_invalid_values=True
                   
                                )[0]
    summary = tokenizer.decode(
                                output_ids.cpu(),
                                skip_special_tokens=True,
                                clean_up_tokenization_spaces=True
                                )
    return summary


In [6]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"])
scores = []
ll = len(validation.values)
for i, article in enumerate(validation.values):
    
    print("\r processing {} from {} with doc length {}".format(i, len(validation), len(article[1].split())), end = "")
#     with tf.device('CPU'):
    generated_summary = generate_summary_pt(article[1])
                                            
    new_score = scorer.score(article[2], generated_summary)
    scores.append(new_score)

    print("\tWith ROUGE scores: " , end= "")
    
    ss = {"rouge1": (sum(s["rouge1"].fmeasure for s in scores ) / len(scores)),
        "rouge2": (sum(s["rouge2"].fmeasure for s in scores) / len(scores)),
        "rougeL": (sum(s["rougeL"].fmeasure for s in scores) / len(scores))}
                   
   
    print(ss)
    
    print('Ground Truth:')
    print('\t\t', article[2])
    print()
    print('Model Prediction')
    print('\t\t', generated_summary)
    print()
    print('='*20)

print()
print('='*50)
avg = {"rouge1": (sum(s["rouge1"].fmeasure for s in scores ) / ll),
        "rouge2": (sum(s["rouge2"].fmeasure for s in scores) / ll),
        "rougeL": (sum(s["rougeL"].fmeasure for s in scores) / ll)}
print('Average ROUGE scores', avg)

 processing 0 from 154 with doc length 335	With ROUGE scores: {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
Ground Truth:
		 يبدأ الكاتب عرض الكتاب الرابع تحت عنوان من الكارثة إلى التحدى؛ حيث يوضح كيف كانت إسرائيل فرحة بنصرها عام 67، وكيف بدأت حرب الاستنزاف، ثم يتكلم عن وفاة عبدالناصر، وتولى أنور السادات حكم مصر، ويعرض الكاتب للخطط والاستعدادات المصرية، ثم الاستعدادات الإسرائيلية، ثم يبدأ بعرض وقائع الحرب، ويتوقف الكاتب عند يوم 8 أكتوبر، ويقول: إن هذا اليوم كان أسوأ هزيمة فى تاريخ الجيش الإسرائيلى، ثم ينتقل بنا إلى الجبهة السورية، ثم يعود ثانية إلى يوميات الحرب من  9 أكتوبر إلى 13 و 14 أكتوبر، والعمليات النهائية فى سوريا 14 إلى 23 أكتوبر، ثم يعرض الكاتب المعركة الخاصة بالاستيلاء على مدينة السويس من 23 أكتوبر إلى 25 أكتوبر، ثم تطورات هذه المعركة. 

Model Prediction
		 ..............................................................................................................................................................................................????.........??????????؟؟؟؟ ؟ ؟ 

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│    5 │                                                                                           │
│    6 │   print("\r processing {} from {} with doc length {}".format(i, len(validation), len(a    │
│    7 #     with tf.device('CPU'):                                                                │
│ ❱  8 │   generated_summary = generate_summary_pt(article[1])                                     │
│    9 │                                                                                           │
│   10 │   new_score = scorer.score(article[2], generated_summary)                                 │
│   11 │   scores.append(new_score)                                                                │
│                                                                                                  │
│ in generate_summary_pt:8                                                                         │
│                                                                                                  │
│    5 │   │   │   │   │   │   return_tensors="pt",                                                │
│    6 │   │   │   │   │   │   )["input_ids"].to(model.device)                                     │
│    7 │   max_output_length = 250                                                                 │
│ ❱  8 │   output_ids = model.generate(                                                            │
│    9 │   │   │   │   │   │   │   │   input_ids=input_ids,                                        │
│   10 │   │   │   │   │   │   │   │   num_beams=8,                                                │
│   11 │   │   │   │   │   │   │   │   length_penalty=2.0,                                         │
│                                                                                                  │
│ /usr/local/lib/python3.8/site-packages/torch/utils/_contextlib.py:115 in decorate_context        │
│                                                                                                  │
│   112 │   @functools.wraps(func)                                                                 │
│   113 │   def decorate_context(*args, **kwargs):                                                 │
│   114 │   │   with ctx_factory():                                                                │
│ ❱ 115 │   │   │   return func(*args, **kwargs)                                                   │
│   116 │                                                                                          │
│   117 │   return decorate_context                                                                │
│   118                                                                                            │
│                                                                                                  │
│ /usr/local/lib/python3.8/site-packages/transformers/generation/utils.py:1611 in generate         │
│                                                                                                  │
│   1608 │   │   │   │   **model_kwargs,                                                           │
│   1609 │   │   │   )                                                                             │
│   1610 │   │   │   # 13. run beam search                                                         │
│ ❱ 1611 │   │   │   return self.beam_search(                                                      │
│   1612 │   │   │   │   input_ids,                                                                │
│   1613 │   │   │   │   beam_scorer,                                                              │
│   1614 │   │   │   │   logits_processor=logits_processor,  